# Part 0 — Tokenization for Health Text

**Guiding question:** How does a sentence of clinical text become the numbers a language model actually reads?

**Purpose:** see that a model never receives words. It receives **tokens**. Watch health vocabulary get split into pieces, and connect that splitting to cost, context limits, and why two spellings of the same concept behave differently.

**Runtime:** about 20 minutes. **Roadmap:** whitespace splitting → the vocabulary problem → build a tokenizer by hand → apply it → compare with a production tokenizer → what this costs you.

**Safety and scope:** all text here is fictional. Never paste patient data, PHI, or identifiers into a tokenizer, a prompt, or a screenshot. Run cells from top to bottom.

## 0. Setup

A **token** is the unit of text a model reads. A **vocabulary** is the fixed list of tokens a model knows. A **tokenizer** is the program that converts text into a sequence of vocabulary entries.

This notebook uses only the standard library plus pandas and matplotlib for display. The optional production tokenizer is loaded later and is not required.

In [ ]:
import re, sys, json
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_colwidth", 80)
print({"python": sys.version.split()[0], "pandas": pd.__version__})

## 1. Start with the obvious idea: split on spaces

The simplest tokenizer splits on whitespace. Let us apply it to fictional clinic text and count what we get.

**Look for:** how many distinct tokens appear, and whether closely related words are treated as related.

In [ ]:
CLINIC_NOTES = [
    "Patient booked a followup appointment through the portal.",
    "Patient booking a follow-up appointment through the portal.",
    "The nurse reviewed the laboratory result and contacted the patient.",
    "The nurse reviews the lab results and contacts the patient.",
    "Metformin 500 mg was continued; metformin 850 mg was not started.",
    "HbA1c was 7.8% at the visit and HbA1c was 8.1% three months earlier.",
]

whitespace_tokens = [note.split() for note in CLINIC_NOTES]
flat = [t for note in whitespace_tokens for t in note]

print("first note ->", whitespace_tokens[0])
print(f"\ntotal tokens: {len(flat)}   distinct tokens: {len(set(flat))}")

### What do we see?

Splitting on spaces produced readable pieces, but it created problems we can already name.

`appointment.` and `appointment` are different strings because punctuation stayed attached. `followup` and `follow-up` are different. `result` and `results` are different. `Metformin` and `metformin` are different. A whitespace tokenizer treats each of these as an unrelated entry.

This is not a small inconvenience. Every distinct string needs its own row in the model's vocabulary and its own learned vector.

## 2. Measure the vocabulary problem directly

Count how many separate vocabulary entries the whitespace tokenizer needs for groups of words that a clinician would consider the same idea.

**Look for:** the `distinct strings` column. Every entry above 1 is a case where the model must learn each form separately.

In [ ]:
concept_groups = {
    "appointment":      [t for t in flat if "appointment" in t.lower()],
    "follow-up":        [t for t in flat if "follow" in t.lower()],
    "laboratory / lab": [t for t in flat if t.lower().startswith(("lab", "laborator"))],
    "result":           [t for t in flat if "result" in t.lower()],
    "metformin":        [t for t in flat if "metformin" in t.lower()],
    "HbA1c":            [t for t in flat if "hba1c" in t.lower()],
}

rows = []
for concept, forms in concept_groups.items():
    distinct = sorted(set(forms))
    rows.append({"clinical concept": concept,
                 "distinct strings": len(distinct),
                 "forms seen": ", ".join(distinct)})

display(pd.DataFrame(rows))

### What do we see?

One clinical idea is scattered across several vocabulary entries. `laboratory` and `lab` share no relationship at all under this scheme, even though a nurse would use them interchangeably.

Two directions could fix this, and both fail on their own:

- **Word-level vocabulary:** keep whole words. English clinical text has an effectively unlimited supply of drug names, abbreviations, and dose strings, so the vocabulary never stops growing. Anything unseen becomes an unknown token and loses all information.
- **Character-level vocabulary:** keep only letters. The vocabulary is tiny and nothing is ever unknown, but a sentence becomes very long and the model must rebuild every word from scratch.

Real tokenizers take the middle path: **subword** units.

## 3. Build a subword tokenizer by hand

**Byte pair encoding (BPE)** learns a vocabulary from data with one repeated rule:

> find the most frequent adjacent pair of symbols, and merge it into a single new symbol.

Start with every word split into characters. Apply the rule a fixed number of times. Each merge is recorded in order, and that ordered list *is* the tokenizer.

The function below is the whole algorithm. Read it before running it.

In [ ]:
def learn_bpe_merges(corpus, num_merges, end_marker="</w>"):
    """Learn an ordered list of BPE merges from a list of sentences."""
    words = [w for sentence in corpus for w in re.findall(r"[a-z0-9]+", sentence.lower())]
    # Each word starts as a tuple of characters plus an end-of-word marker.
    vocabulary = Counter(tuple(word) + (end_marker,) for word in words)

    merges = []
    for step in range(num_merges):
        pair_counts = Counter()
        for symbols, count in vocabulary.items():
            for left, right in zip(symbols, symbols[1:]):
                pair_counts[(left, right)] += count
        if not pair_counts:
            break
        # Sort by count, then alphabetically, so the result is reproducible.
        best_pair, best_count = max(sorted(pair_counts.items()), key=lambda kv: kv[1])
        merges.append({"step": step + 1, "pair": best_pair,
                       "merged": "".join(best_pair), "count": best_count})

        merged_vocabulary = Counter()
        for symbols, count in vocabulary.items():
            merged_vocabulary[apply_one_merge(symbols, best_pair)] += count
        vocabulary = merged_vocabulary
    return merges, vocabulary


def apply_one_merge(symbols, pair):
    """Replace every adjacent occurrence of `pair` in `symbols` with the joined symbol."""
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)


print("Two functions defined: learn_bpe_merges and apply_one_merge.")

## 4. Train it on fictional clinic text and watch the merges

Run 60 merges over a small fictional corpus. The table shows the first merges in the order they were learned.

**Look for:** early merges are common letter pairs. Later merges are recognisable word fragments. Nobody told the algorithm what a word is.

> ### Your turn
>
> The next code cell is incomplete. Replace every `# TODO` with working code.
> The surrounding explanation tells you what the cell must do and what to look for.
> If you get stuck, the reasoning you need is in the Markdown directly above.

In [ ]:
# TODO: add three sentences of your own fictional clinic text to TRAINING_CORPUS
# below, then choose a number of merges. Try 60 first, then try 20 and 200 and
# compare which words become single tokens.
NUM_MERGES = 60  # TODO: experiment with this

merges, final_vocabulary = learn_bpe_merges(TRAINING_CORPUS, num_merges=NUM_MERGES)

merge_table = pd.DataFrame([
    {"step": m["step"], "merged pair": f'{m["pair"][0]!r} + {m["pair"][1]!r}',
     "new symbol": m["merged"], "times seen": m["count"]}
    for m in merges
])
print(f"learned {len(merges)} merges")
display(merge_table.head(18))

### What do we see?

The first merges join very common letter pairs. By the later steps the algorithm has assembled fragments such as `re`, `ti`, `on`, and eventually whole common words.

The important property: **frequent things become single tokens, rare things stay in pieces.** The algorithm spends its limited vocabulary budget on what the corpus actually contains. A corpus of clinic scheduling text will learn `appointment` as one unit. It will not learn `myocardial`.

## 5. Apply the learned tokenizer to new words

Encoding applies the learned merges to a new word **in the order they were learned**. Words the corpus contained collapse into few tokens. Words it never saw stay fragmented.

**Look for:** the `tokens` count for words that were in the training corpus versus words that were not.

> ### Your turn
>
> The next code cell is incomplete. Replace every `# TODO` with working code.
> The surrounding explanation tells you what the cell must do and what to look for.
> If you get stuck, the reasoning you need is in the Markdown directly above.

In [ ]:
corpus_words = [w for s in TRAINING_CORPUS for w in re.findall(r"[a-z0-9]+", s.lower())]
corpus_counts = Counter(corpus_words)

# TODO: add at least three clinical terms of your own to this list.
# Predict how many tokens each will need BEFORE you run the cell, then check.
probe_words = ["the", "portal", "result", "appointment", "laboratory",
               "metformin", "hba1c", "myocardial", "tachycardia"]

rows = []
for word in probe_words:
    pieces = encode_with_bpe(word, merges)
    rows.append({"word": word,
                 "times in training corpus": corpus_counts.get(word.lower(), 0),
                 "tokens": len(pieces),
                 "pieces": " | ".join(pieces)})

probe_table = pd.DataFrame(rows).sort_values("times in training corpus", ascending=False)
display(probe_table.reset_index(drop=True))

### What do we see?

Sort the table by frequency and the pattern is immediate: **how often a word appeared decides how few tokens it costs.**

`portal`, `result`, and `appointment` occur repeatedly, so merges were spent on them and each collapses to a single token. `metformin` and `hba1c` *are* in the training text, but only once or twice — not often enough to win a merge — so they stay fragmented. `myocardial` and `tachycardia` never appeared at all and break down almost to single characters.

Presence in the corpus is not what matters. **Frequency** is. A tokenizer has a fixed vocabulary budget and spends it on what it sees most.

Note also that nothing is ever *unknown*: an unseen word still encodes, just expensively. That is the advantage of subwords over a fixed word list.

## 6. Compare with the tokenizer used by a production model

The hand-built tokenizer above used 60 merges and 14 sentences. A production tokenizer is trained on a very large corpus and has roughly 100,000 entries.

The next cell loads `cl100k_base`, the tokenizer family used by the models in Part 3. It needs to download a vocabulary file the first time it runs. If that is unavailable, the notebook says so and continues — the teaching points above do not depend on it.

Documentation: [tiktoken](https://github.com/openai/tiktoken).

In [ ]:
PRODUCTION_TOKENIZER = None
try:
    import tiktoken
    PRODUCTION_TOKENIZER = tiktoken.get_encoding("cl100k_base")
    print(f"loaded cl100k_base | vocabulary size: {PRODUCTION_TOKENIZER.n_vocab:,}")
except Exception as error:
    print("Production tokenizer unavailable in this environment.")
    print(f"Reason: {type(error).__name__}. Sections 6 and 7 will be skipped; the rest of the notebook still runs.")

### Tokenize a realistic clinical sentence

**Look for:** where the boundaries fall inside drug names, laboratory abbreviations, and numbers. Each box in the output is one token.

In [ ]:
CLINICAL_SENTENCE = ("Patient presented with myocardial infarction; metformin 0.125 mg daily. "
                     "HbA1c 7.8%. Follow-up in the portal within three business days.")

if PRODUCTION_TOKENIZER is None:
    print("Skipped: production tokenizer not loaded.")
else:
    ids = PRODUCTION_TOKENIZER.encode(CLINICAL_SENTENCE)
    pieces = [PRODUCTION_TOKENIZER.decode([i]) for i in ids]
    print(f"characters: {len(CLINICAL_SENTENCE)}   tokens: {len(ids)}\n")
    print(" | ".join(piece.replace(" ", "_") for piece in pieces))
    print("\n(_ marks a leading space that belongs to the token.)")

### What do we see?

Read the boundaries carefully, because this is the central observation of the notebook.

- `myocardial` is not one token. It becomes `myocard` + `ial`.
- `infarction` becomes `inf` + `ar` + `ction`.
- `metformin` becomes `met` + `form` + `in`. The fragment `form` is the same fragment that appears in the ordinary word *form*.
- `HbA1c` becomes five separate tokens, roughly one per character.
- `0.125` and `7.8` are split around the decimal point.

Ordinary English words cost about one token each. Clinical vocabulary costs several. The model still works with these terms, but it processes them as assemblies of fragments rather than as single units.

## 7. Count the cost: tokens drive price and context limits

Two practical consequences follow directly from the token count:

1. **Cost.** Hosted models bill per token. Clinical text costs more per word than general prose.
2. **Context limit.** A model accepts a fixed maximum number of tokens. In Part 3 this decides how many retrieved chunks fit in a prompt.

**Look for:** the `tokens per word` column. Compare the clinical rows with the plain-English row.

In [ ]:
SAMPLES = {
    "plain English":        "The patient will come back to the clinic next week for a check.",
    "clinic admin text":    "Routine laboratory results are posted to the portal within three business days.",
    "clinical narrative":   "Myocardial infarction with subsequent percutaneous coronary intervention.",
    "medication list":      "Metformin 500 mg BID; atorvastatin 40 mg daily; lisinopril 10 mg daily.",
    "lab panel":            "HbA1c 7.8%, eGFR 62 mL/min/1.73m2, LDL-C 2.4 mmol/L, TSH 3.1 mIU/L.",
}

if PRODUCTION_TOKENIZER is None:
    print("Skipped: production tokenizer not loaded.")
else:
    rows = []
    for label, text in SAMPLES.items():
        n_tokens = len(PRODUCTION_TOKENIZER.encode(text))
        n_words = len(text.split())
        rows.append({"text type": label, "words": n_words, "tokens": n_tokens,
                     "tokens per word": round(n_tokens / n_words, 2)})
    cost_table = pd.DataFrame(rows)
    display(cost_table)

    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.barh(cost_table["text type"], cost_table["tokens per word"], color="#0072B2")
    ax.axvline(1.0, color="black", ls="--", lw=1)
    ax.set(title="Tokens per word by text type (dashed line = one token per word)",
           xlabel="Tokens per word", ylabel="")
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout(); plt.show()

### What do we see?

Plain English sits near one token per word. A laboratory panel can be two to three times higher.

Two consequences to carry into Part 3:

- A budget estimated from general prose will **understate** the cost of clinical text.
- A context window described as "about 3,000 words" holds considerably less clinical text than that.

The number of tokens is a property of the text and the tokenizer. It says nothing about whether the content is correct.

## 8. The consequence that matters most for retrieval

Tokenization also explains something you will meet in Part 3.

`lab result`, `laboratory result`, and `bloodwork` mean the same thing to a clinician. To a tokenizer they are different token sequences with no shared fragment carrying that meaning.

**Look for:** the number of shared tokens between pairs that mean the same thing.

In [ ]:
SYNONYM_PAIRS = [
    ("laboratory result", "lab result"),
    ("laboratory result", "bloodwork"),
    ("appointment", "visit"),
    ("proxy access", "caregiver access"),
]

if PRODUCTION_TOKENIZER is None:
    print("Skipped: production tokenizer not loaded.")
else:
    rows = []
    for left, right in SYNONYM_PAIRS:
        left_ids = set(PRODUCTION_TOKENIZER.encode(left))
        right_ids = set(PRODUCTION_TOKENIZER.encode(right))
        rows.append({"phrase A": left, "phrase B": right,
                     "tokens A": len(left_ids), "tokens B": len(right_ids),
                     "shared tokens": len(left_ids & right_ids)})
    display(pd.DataFrame(rows))

### What do we see?

Phrases that a clinician treats as equivalent often share no tokens at all.

This is exactly why Part 3 adds a **terminology file**. Matching on tokens or on characters cannot know that `bloodwork` and `laboratory result` are the same concept. Something has to state that relationship explicitly, or the system has to learn it from context — which is what Part 3 covers next.

## 9. Learner checks

1. **Why does a subword tokenizer never produce an unknown token?** Because it can always fall back to single characters, which are all in the vocabulary.
2. **A term is split into six tokens. What does that tell you about the training corpus?** The term was rare in it. It tells you nothing about whether the term is clinically important.
3. **You budget an API call using a word count from an English essay. What happens with a laboratory panel?** You underestimate the cost, because clinical text uses more tokens per word.
4. **Does a low token count mean the text is accurate?** No. Token count measures length under a tokenizer, nothing else.
5. **Why can two phrases with identical meaning share zero tokens?** Tokens are learned from character frequency, not from meaning.

## Summary, limitations, and completion checklist

**Result:** text reaches a model as a sequence of subword tokens. The split is learned from character frequency in a training corpus, so common words become single tokens and rare clinical terms become several fragments.

- [x] Showed that whitespace splitting fragments one concept across many vocabulary entries.
- [x] Implemented byte pair encoding and inspected the learned merges.
- [x] Applied it to words inside and outside the training corpus.
- [x] Compared with the production tokenizer used by the Part 3 models.
- [x] Measured tokens per word across clinical text types.
- [x] Showed that synonymous clinical phrases can share no tokens.

**Limits:** the hand-built tokenizer is a teaching model, not a production one — it ignores bytes, casing rules, special tokens, and regex pre-splitting. Token counts here reflect one tokenizer at one point in time. Nothing in this notebook evaluates whether text is clinically correct.

**Try it yourself:** add three sentences of your own fictional clinic text to `TRAINING_CORPUS`, rerun the merge cell, and check whether any of your new words became a single token.

Next: `Part-1_Word2Vec_and_tSNE.ipynb`, which asks how a token gets a *meaning* rather than just an identity.